# Hướng dẫn Tinh chỉnh (Fine-tuning) PaddleOCR Recognition cho Mã Container trên Google Colab

Notebook này hỗ trợ bạn kết nối Google Drive, giải nén và tự động cắt (crop) dữ liệu từ tập ảnh thô dựa trên nhãn tọa độ, tự động chia bộ dữ liệu Train/Val theo tỉ lệ 90/10, cấu hình và huấn luyện tinh chỉnh mô hình nhận dạng ký tự **PP-OCRv3** của PaddleOCR chuyên biệt cho phông chữ Container, sau đó xuất ra mô hình suy luận tĩnh (`inference model`) lưu về Google Drive của bạn.

## Bước 1: Kết nối Google Drive và GPU

In [1]:
# 1. Kết nối tới Google Drive của bạn
from google.colab import drive
drive.mount('/content/drive')

# 2. Kiểm tra thông tin GPU
!nvidia-smi

Mounted at /content/drive
Thu Aug 20 09:11:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

## Bước 2: Cài đặt thư viện PaddlePaddle GPU và PaddleOCR

In [15]:
# 1. Cài đặt thư viện PaddlePaddle hỗ trợ GPU (phù hợp với CUDA trên Colab)
# Thay thế lệnh cũ ở Bước 2 bằng lệnh này:
!pip install paddlepaddle-gpu==2.6.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu120/

# 2. Clone mã nguồn PaddleOCR
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR

# 3. Cài đặt các thư viện phụ thuộc
!pip install -r requirements.txt

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu120/
Cloning into 'PaddleOCR'...
remote: Enumerating objects: 353017, done.
remote: Counting objects: 100% (1276/1276), done.
remote: Compressing objects: 100% (177/177), done.
^C
[Errno 2] No such file or directory: 'PaddleOCR'
/content/PaddleOCR
Ignoring lmdb: markers 'python_version < "3.9"' don't match your environment
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^

## Bước 3: Tự động Giải nén và Xử lý cắt (Crop) dữ liệu ảnh container

Đoạn mã dưới đây sẽ tự động thực hiện:
1. Giải nén tệp `ContainerNum_dataset.zip` từ Drive của bạn.
2. Giải nén các tệp zip lồng nhau (`train_images.zip`, `train_images_label.zip`).
3. Đọc nhãn tọa độ (`x1, y1, x2, y2, nhãn_chữ`) từ các tệp `.txt` gán nhãn.
4. Tự động cắt (crop) ảnh chứa dòng mã container từ ảnh thô lớn.
5. Tự động chia tập dữ liệu thành 90% Train và 10% Validation.
6. Tạo các tệp nhãn `rec_train_label.txt` và `rec_val_label.txt` chuẩn định dạng của PaddleOCR.

In [3]:
import os
import zipfile
import io
import cv2
import re
import random

# 1. Đường dẫn file ZIP trên Google Drive
outer_zip_path = "/content/drive/MyDrive/ContainerNum_dataset.zip"

print("Đang giải nén file ZIP chính...")
with zipfile.ZipFile(outer_zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/temp_dataset")

# 2. Đường dẫn các zip con vừa được giải nén ra
train_images_zip = "/content/temp_dataset/ContainerNum_dataset/train_images.zip"
train_labels_zip = "/content/temp_dataset/ContainerNum_dataset/train_images_label.zip"

print("Đang giải nén ảnh thô và nhãn tọa độ...")
with zipfile.ZipFile(train_images_zip, 'r') as zip_ref:
    zip_ref.extractall("/content/temp_train_images")

with zipfile.ZipFile(train_labels_zip, 'r') as zip_ref:
    zip_ref.extractall("/content/temp_train_labels")

# 3. Tạo cấu trúc thư mục huấn luyện
os.makedirs("/content/PaddleOCR/train_data/rec/train", exist_ok=True)
os.makedirs("/content/PaddleOCR/train_data/rec/val", exist_ok=True)

# Quét tất cả file nhãn dạng text
label_dir = "/content/temp_train_labels/images_label"
label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]

# Chia tập Train (90%) và Val (10%)
random.seed(42)
random.shuffle(label_files)
split_idx = int(len(label_files) * 0.9)
train_files = label_files[:split_idx]
val_files = label_files[split_idx:]

def process_and_crop(files, subset):
    label_entries = []
    count = 0
    for lf in files:
        base_name = os.path.splitext(lf)[0]
        img_name = base_name + ".jpg"

        img_path = os.path.join("/content/temp_train_images/images", img_name)
        label_path = os.path.join(label_dir, lf)

        if not os.path.exists(img_path):
            continue

        # Đọc ảnh
        img = cv2.imread(img_path)
        if img is None:
            continue

        # Đọc nhãn tọa độ dạng x1,y1,x2,y2,label
        with open(label_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        for idx, line in enumerate(lines):
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            if len(parts) >= 5:
                try:
                    # Ép kiểu tọa độ
                    x1 = int(float(parts[0]))
                    y1 = int(float(parts[1]))
                    x2 = int(float(parts[2]))
                    y2 = int(float(parts[3]))
                    label = ",".join(parts[4:]).upper()

                    # Làm sạch nhãn chữ (chỉ giữ ký tự và số)
                    label = re.sub(r'[^A-Z0-9]', '', label)
                    if not label:
                        continue

                    h_orig, w_orig, _ = img.shape
                    x1_c, y1_c = max(0, x1), max(0, y1)
                    x2_c, y2_c = min(w_orig, x2), min(h_orig, y2)

                    if x2_c <= x1_c or y2_c <= y1_c:
                        continue

                    # Cắt vùng chữ
                    crop = img[y1_c:y2_c, x1_c:x2_c]

                    # Lưu ảnh crop
                    crop_name = f"{base_name}_{idx}.jpg"
                    save_path = os.path.join(f"/content/PaddleOCR/train_data/rec/{subset}", crop_name)
                    cv2.imwrite(save_path, crop)

                    # Tạo chuỗi nhãn chuẩn
                    relative_path = f"train_data/rec/{subset}/{crop_name}"
                    label_entries.append(f"{relative_path}\t{label}\n")
                    count += 1
                except Exception as e:
                    pass

    # Ghi file danh sách nhãn
    label_txt_path = f"/content/PaddleOCR/train_data/rec_{subset}_label.txt"
    with open(label_txt_path, 'w', encoding='utf-8') as f:
        f.writelines(label_entries)
    print(f"-> Phân tập '{subset}': Đã cắt và lưu thành công {count} ảnh mã container.")

process_and_crop(train_files, "train")
process_and_crop(val_files, "val")

# Dọn dẹp bộ nhớ tạm giải phóng không gian ổ đĩa Colab
!rm -rf /content/temp_dataset /content/temp_train_images /content/temp_train_labels
print("Đã dọn dẹp các thư mục trung gian thành công!")

Đang giải nén file ZIP chính...
Đang giải nén ảnh thô và nhãn tọa độ...
-> Phân tập 'train': Đã cắt và lưu thành công 4975 ảnh mã container.
-> Phân tập 'val': Đã cắt và lưu thành công 566 ảnh mã container.
Đã dọn dẹp các thư mục trung gian thành công!


## Bước 4: Tải Trọng số Pre-trained Model PP-OCRv3 English

In [4]:
# Tải mô hình pre-trained PP-OCRv3 tiếng Anh
!mkdir -p pretrain_models
!wget -P pretrain_models/ https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_rec_train.tar

# Giải nén gói trọng số
%cd pretrain_models
!tar -xf en_PP-OCRv3_rec_train.tar
%cd ..

--2026-08-20 09:18:32--  https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_rec_train.tar
Resolving paddleocr.bj.bcebos.com (paddleocr.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:913:0:ff:b0a4:a156
Connecting to paddleocr.bj.bcebos.com (paddleocr.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 204093440 (195M) [application/x-tar]
Saving to: ‘pretrain_models/en_PP-OCRv3_rec_train.tar’

en_PP-OCRv3_rec_tra 100%[===================>] 194.64M  14.4MB/s    in 39s     

2026-08-20 09:19:15 (4.93 MB/s) - ‘pretrain_models/en_PP-OCRv3_rec_train.tar’ saved [204093440/204093440]

/content/PaddleOCR/pretrain_models
/content/PaddleOCR


## Bước 5: Cấu hình File YAML để bắt đầu Training

Đoạn mã Python dưới đây tự động cập nhật các trường đường dẫn dữ liệu huấn luyện, đường dẫn trọng số pre-trained, số epoch chạy và cấu hình từ điển vào tệp config `en_PP-OCRv3_rec.yml` của PaddleOCR.

In [6]:
import yaml

config_path = '/content/PaddleOCR/configs/rec/PP-OCRv3/PP-OCRv3_mobile_rec.yml'

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# 1. Ghi đè các thông số cấu hình cốt lõi dùng đường dẫn tuyệt đối
config['Global']['pretrained_model'] = '/content/PaddleOCR/pretrain_models/en_PP-OCRv3_rec_train/best_accuracy'
config['Global']['save_model_dir'] = '/content/PaddleOCR/output/v3_rec_container/'
config['Global']['epoch_num'] = 150              # Huấn luyện 150 Epochs
config['Global']['print_batch_step'] = 10
config['Global']['use_gpu'] = True

# 2. Cấu hình đường dẫn dữ liệu Training
config['Train']['dataset']['data_dir'] = '/content/PaddleOCR/train_data/rec/train/'
config['Train']['dataset']['label_file_list'] = ['/content/PaddleOCR/train_data/rec_train_label.txt']

# 3. Cấu hình đường dẫn dữ liệu Evaluation
config['Eval']['dataset']['data_dir'] = '/content/PaddleOCR/train_data/rec/val/'
config['Eval']['dataset']['label_file_list'] = ['/content/PaddleOCR/train_data/rec_val_label.txt']

# Lưu lại cấu hình mới vào đè lên file
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, default_flow_style=False)

print("Đã ghi đè cấu hình file YAML huấn luyện thành công!")

Đã ghi đè cấu hình file YAML huấn luyện thành công!


## Bước 6: Khởi chạy Huấn luyện Tinh chỉnh (Fine-tuning)

In [14]:
import yaml
import os

# Đường dẫn đến file cấu hình YAML
config_path = '/content/PaddleOCR/configs/rec/PP-OCRv3/PP-OCRv3_mobile_rec.yml'

# Đọc cấu hình hiện tại
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Cập nhật đường dẫn data_dir để PaddleOCR có thể tìm thấy ảnh
# Lỗi xuất hiện vì data_dir được thiết lập là thư mục con, nhưng các nhãn
# lại chứa đường dẫn tương đối từ thư mục gốc của PaddleOCR.
# Thay đổi data_dir thành thư mục gốc của PaddleOCR.
config['Train']['dataset']['data_dir'] = '/content/PaddleOCR/'
config['Eval']['dataset']['data_dir'] = '/content/PaddleOCR/'

# Cập nhật đường dẫn từ điển ký tự để khớp với mô hình tiếng Anh đã pre-trained
config['Global']['character_dict_path'] = '/content/PaddleOCR/ppocr/utils/en_dict.txt'

# Giảm num_workers về 0 để debug lỗi Segmentation Fault, thường liên quan đến data loading
config['Train']['loader']['num_workers'] = 0
config['Eval']['loader']['num_workers'] = 0

# Thay thế toàn bộ danh sách 'transforms' bằng một tập hợp cơ bản, không có RecConAug,
# để khắc phục lỗi Segmentation Fault dai dẳng do các biến đổi dữ liệu phức tạp.
common_transforms = [
    {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
    {'MultiLabelEncode': None}, # Cần cho SAR head
    {'RecResizeImg': {'image_shape': [3, 48, 320]}},
    {'KeepKeys': {'keep_keys': ['image', 'label_ctc', 'label_sar', 'length', 'valid_ratio']}}
]

config['Train']['dataset']['transforms'] = common_transforms
config['Eval']['dataset']['transforms'] = common_transforms


# Lưu lại cấu hình đã chỉnh sửa vào file
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, default_flow_style=False)

print("Đã điều chỉnh lại cấu hình file YAML huấn luyện để khắc phục lỗi đường dẫn ảnh, từ điển ký tự, num_workers và đảm bảo loại bỏ RecConAug.")

# Chạy lệnh python huấn luyện dùng đường dẫn tuyệt đối
!python /content/PaddleOCR/tools/train.py -c {config_path}

Đã điều chỉnh lại cấu hình file YAML huấn luyện để khắc phục lỗi đường dẫn ảnh, từ điển ký tự, num_workers và đảm bảo loại bỏ RecConAug.
Skipping import of the encryption module.
[2026/08/20 09:30:27] ppocr INFO: Architecture : 
[2026/08/20 09:30:27] ppocr INFO:     Backbone : 
[2026/08/20 09:30:27] ppocr INFO:         last_conv_stride : [1, 2]
[2026/08/20 09:30:27] ppocr INFO:         last_pool_kernel_size : [2, 2]
[2026/08/20 09:30:27] ppocr INFO:         last_pool_type : avg
[2026/08/20 09:30:27] ppocr INFO:         name : MobileNetV1Enhance
[2026/08/20 09:30:27] ppocr INFO:         scale : 0.5
[2026/08/20 09:30:27] ppocr INFO:     Head : 
[2026/08/20 09:30:27] ppocr INFO:         head_list : 
[2026/08/20 09:30:27] ppocr INFO:             CTCHead : 
[2026/08/20 09:30:27] ppocr INFO:                 Head : 
[2026/08/20 09:30:27] ppocr INFO:                     fc_decay : 1e-05
[2026/08/20 09:30:27] ppocr INFO:                 Neck : 
[2026/08/20 09:30:27] ppocr INFO:                 

## Bước 7: Xuất Mô hình Suy luận tĩnh (Inference Model) về Google Drive

Khi quá trình training kết thúc, tệp trọng số tốt nhất được lưu tại `/content/PaddleOCR/output/v3_rec_container/best_accuracy`. Ta sẽ xuất nó sang định dạng suy luận tĩnh rồi lưu trực tiếp vào Google Drive để bạn dễ dàng tải về tích hợp vào code chạy ứng dụng Web UI offline!

In [ ]:
# 1. Định nghĩa thư mục lưu trữ trên Google Drive của bạn
drive_output_dir = '/content/drive/MyDrive/paddle_rec_inference/'

# 2. Chạy script export model của PaddleOCR dùng đường dẫn tuyệt đối
!python /content/PaddleOCR/tools/export_model.py \
  -c /content/PaddleOCR/configs/rec/PP-OCRv3/en_PP-OCRv3_rec.yml \
  -o Global.pretrained_model=/content/PaddleOCR/output/v3_rec_container/best_accuracy \
  Global.save_inference_dir={drive_output_dir}

print(f"Xuất mô hình thành công! Bạn có thể lấy tệp tại thư mục: {drive_output_dir} trên Google Drive.")